# Vogen Training Notebook

This Colab notebook demonstrates training the Vogen environment using Unsloth and TRL.

## Setup

First, let's install the required dependencies and clone the repository.

In [ ]:
# Install system dependencies
!apt-get update && apt-get install -y git

# Install Python packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth trl transformers accelerate peft datasets
!pip install openenv-core fastapi uvicorn requests pyyaml matplotlib
!pip install wandb  # optional for logging

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/vogen.git
%cd vogen

# Install the package
!pip install -e .

## Start the Environment Server

We need to start the FastAPI server in the background to provide the environment for training.

In [ ]:
# Start the server in background
import subprocess
import time

server_process = subprocess.Popen(['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(5)  # Wait for server to start
print("Server started on port 8000")

## Training Configuration

Let's set up the training configuration. For Colab, we'll use a smaller model and fewer episodes.

In [ ]:
import yaml

# Colab-friendly config
config = {
    'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
    'train_episodes': 5,  # Reduced for Colab
    'max_steps': 3,
    'lora_r': 16,
    'lora_target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    'batch_size': 1,
    'learning_rate': 1e-5,
    'num_train_epochs': 1,
    'reward_weights': {
        'critic_score': 1.0,
        'justification': 0.4,
        'novelty': 0.3,
        'difficulty': 0.5,
        'calibration': 0.2
    }
}

# Save config
with open('training/configs/grpo_colab.yaml', 'w') as f:
    yaml.dump(config, f)

print("Config saved to training/configs/grpo_colab.yaml")

## Run Training

Now let's run the training script. This will generate real training data from the environment and train the model.

In [ ]:
# Run training
!python -m training.train_vogen --config training/configs/grpo_colab.yaml

# Generate plots
!python -m training.train_with_plots --config training/configs/grpo_colab.yaml --episodes 5 --output-dir results

## View Results

Let's check the generated plots and metrics.

In [ ]:
import matplotlib.pyplot as plt
import json

# Load metrics
with open('results/metrics.json') as f:
    metrics = json.load(f)

print("Training Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

# Display plots
from IPython.display import Image
Image('results/reward_curve.png')
# You can add more Image() calls for other plots

## Cleanup

Stop the server when done.

In [ ]:
# Stop server
server_process.terminate()
server_process.wait()
print("Server stopped")